## Apply Savitzky-Golay Filter on DEA FC data

In [1]:
### Essential Libary Imports
import numpy as np
import pandas as pd
from datetime import datetime

import sys
import os 
from sklearn.pipeline import Pipeline

# Import Custom Module
sys.path.append('C:/Users/Krish Singh (Work)/Desktop/MRes_Publication_Repository_KKS/in-situ_vs_rs_paper/ANCILLARY_SCRIPTS')
from Preprocess_Data import * # import from custom transformers 

In [2]:
## Create a list of DEA FC files that are currently present in the directory 
files = os.listdir(f'../DATASETS/DEA_FC_PROCESSED/SPATIAL_AND_UE_FILTER/')
files = [i.split('.')[0] for i in files]
all_dea_fc = files[1:]

In [3]:
## Savitzky-Golay Filter Parameters:
window_length_smooth = 15
polyorder = 4 

In [4]:
## Main Loop where we are going through through the list of datafiles,
##  reading them in, applying the Savitzky-Golay Filter, and saving them to an output,
##  all in an iterative fashion
max_counter = len(all_dea_fc)
counter = 1
print(f'The Data Awaiting Preprocessing: {all_dea_fc}')

for site_location_name in all_dea_fc:
    print(f'Processing Data for {site_location_name}, {counter}/{max_counter}')
    
    # Read in the File 
    site = pd.read_csv(f'C:/Users/Krish Singh (Work)/Desktop/MRes_Publication_Repository_KKS/in-situ_vs_rs_paper/DATASETS/DEA_FC_PROCESSED/SPATIAL_AND_UE_FILTER/{site_location_name}.csv', parse_dates=['time'])
    
    # The Savitzky-Golay Filter pipeline 
    time_fc_pipeline = Pipeline([
        ('preprocess_fc_time_series', 
        preprocess_fc_time_series(window_length = window_length_smooth, polyorder = polyorder))
    ])
    site_resampled = time_fc_pipeline.fit_transform(site)
    
    # Export the processed dataset 
    site_merged = site_resampled.copy()
    site_merged.to_csv(f'C:/Users/Krish Singh (Work)/Desktop/MRes_Publication_Repository_KKS/in-situ_vs_rs_paper/DATASETS/DEA_FC_PROCESSED/MODELLED_PREPROCESSED/Input_DataSet_{site_location_name}.csv')
    counter += 1

The Data Awaiting Preprocessing: ['NSABBS0001', 'NSABBS0002', 'NSABBS0003', 'NSABBS0004', 'NSABBS0005', 'NSABBS0006', 'NSABHC0001', 'NSABHC0002', 'NSABHC0003', 'NSABHC0004', 'NSABHC0005', 'NSABHC0006', 'NSABHC0007', 'NSABHC0008', 'NSABHC0009', 'NSABHC0010', 'NSABHC0011', 'NSABHC0012', 'NSABHC0013', 'NSABHC0014', 'NSABHC0015', 'NSABHC0016', 'NSABHC0017', 'NSABHC0018', 'NSABHC0019', 'NSABHC0020', 'NSABHC0021', 'NSABHC0022', 'NSABHC0023', 'NSABHC0024', 'NSABHC0025', 'NSABHC0026', 'NSABHC0028', 'NSABHC0029', 'NSACHC0001', 'NSACHC0002', 'NSACHC0003', 'NSACHC0004', 'NSACOP0001', 'NSACOP0002', 'NSAMDD0001', 'NSAMDD0002', 'NSAMDD0003', 'NSAMDD0004', 'NSAMDD0005', 'NSAMDD0006', 'NSAMDD0007', 'NSAMDD0008', 'NSAMDD0009', 'NSAMDD0010', 'NSAMDD0011', 'NSAMDD0012', 'NSAMDD0013', 'NSAMDD0014', 'NSAMDD0015', 'NSAMDD0016', 'NSAMDD0017', 'NSAMDD0018', 'NSAMDD0019', 'NSAMDD0020', 'NSAMDD0021', 'NSAMDD0022', 'NSAMDD0023', 'NSAMDD0024', 'NSAMDD0025', 'NSAMDD0026', 'NSAMDD0027', 'NSAMDD0028', 'NSAMDD0029', 

## Sanity checks against old processing methods 

In [ ]:
## Sanity Checks 

## Check if the difference is mainly to do with leap years 

for site_location_name in all_dea_fc:
    
    new_file = pd.read_csv(f"C:/Users/Krish Singh (Work)/Desktop/MRes_Publication_Repository_KKS/in-situ_vs_rs_paper/DATASETS/DEA_FC_PROCESSED/MODELLED_PREPROCESSED/Input_DataSet_{site_location_name}.csv")
    old_file = pd.read_csv(f"C:/Users/Krish Singh (Work)/Desktop/MRes_Publication_Repository_KKS/in-situ_vs_rs_paper/DATASETS/DEA_FC_PROCESSED/MODELLED_PREPROCESSED_OLD/Input_DataSet_{site_location_name}.csv")
    difference_date_set = set(new_file.time).difference(set(old_file.time))
    
    if difference_date_set:
        for date in difference_date_set:
            date_obj = datetime.strptime(date, '%Y-%m-%d')
            print(f'Leap Year ({date_obj.year}) = {date_obj.year % 4 == 0}')
    
    # Results seem to suggest leap-year handling was the cause of the length of the time series difference
    # But, now I should check if the savitzky-golay filter values are identical (they should be), if YYYY-12-31
    #   dates were excluded post filtering

In [ ]:
## A test to see if we get identical result, even if we 

columns_of_interest = ['time', 'pv_filter']
passes = {}

for site_location_name in all_dea_fc:
    
    new_file = pd.read_csv(f"C:/Users/Krish Singh (Work)/Desktop/MRes_Publication_Repository_KKS/in-situ_vs_rs_paper/DATASETS/DEA_FC_PROCESSED/MODELLED_PREPROCESSED/Input_DataSet_{site_location_name}.csv",
                           usecols= columns_of_interest)
    old_file = pd.read_csv(f"C:/Users/Krish Singh (Work)/Desktop/MRes_Publication_Repository_KKS/in-situ_vs_rs_paper/DATASETS/DEA_FC_PROCESSED/MODELLED_PREPROCESSED_OLD/Input_DataSet_{site_location_name}.csv",
                           usecols= columns_of_interest)

    old_file = old_file.merge(new_file, left_on= 'time', right_on = 'time')
    passes[site_location_name] = bool(np.all(old_file['pv_filter_x'] == old_file['pv_filter_y']))
